# 📦 EC3. Rentabilidad de Proyectos de Calidad en "Bio-Distri S.A."

**Material desarrollado por:** Javier Morales, equipo [IA4LEGOS](https://ia4legos.umh.es/)

**Tema:** Generación de variables aleatorias.

**Licencia:** [CC BY-SA 4.0](http://creativecommons.org/licenses/by-sa/4.0/)

No olvides hacer una copia de este cuaderno (`Archivo > Guardar una copia en Drive`) antes de empezar a trabajar.



In [ ]:
#%%capture
# @title ⚠️ Cargar configuración del cuaderno
# Cargamos módulos de análisis numérico
import numpy as np          # importamos numpy como np
import pandas as pd         # importamos pandas como pd
import math                 # importamos módulo para cáculos matemáticos
import random
import inspect
from scipy import stats
import matplotlib.colors as mcolors # Importamos matplotlib.colors

# Cargamos módulos de análisis gráficos
from plotnine import *      # importamos módulo para gráficos con ggplot
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
%config InlineBackend.figure_format = 'retina'

#===============================================
# Cargar funciones bloque1
from urllib.request import urlretrieve
import re # for text manipulation


#url = 'https://raw.githubusercontent.com/asunmayoral/umh1477/refs/heads/main/bloque1_sps.py'
url = 'https://raw.githubusercontent.com/UMH1477/python/refs/heads/main/bloque1_sps.py'
urlretrieve(url, "bloque1_sps.py")
import bloque1_sps
from bloque1_sps import *

# visualización de las funciones precargadas
functions = []
for name, obj in inspect.getmembers(bloque1_sps):
    if inspect.isfunction(obj) and obj.__module__ == bloque1_sps.__name__:
        functions.append(name)

print("\nFunciones precargadas en bloque1_sps.py:\n")
for func_name in functions:
    print(f"- {func_name}")

# 1. Introducción

La distribuidora **"Bio-Distri S.A."** transporta y almacena alimentos frescos a través de una red de centros logísticos. Su principal problema operativo es la **merma**: el alimento que se estropea o se desperdicia antes de llegar al cliente. Para reducirla, la dirección financia en cada centro logístico un **proyecto de mejora de calidad**, pero su naturaleza es incierta: la inversión necesaria, la reducción de merma que realmente se consigue y el ahorro operativo adicional que genera dependen del tipo de proyecto y de factores que solo se conocen una vez ejecutado.

Como analista cuantitativo de la empresa, tu misión es simular el impacto financiero de estos proyectos para poder decidir cuál es la estrategia más rentable y más robusta frente al riesgo.

Este estudio de caso está organizado en dos bloques:

* **Bloque Básico** modela la rentabilidad de **un único proyecto** en un centro logístico, con un horizonte de un año.
* **Bloque Avanzado** convierte ese proyecto puntual en un **programa de mejora continua a varios años**: la eficacia del proyecto mejora con la experiencia acumulada (curva de aprendizaje), pero el ahorro conseguido se degrada con el tiempo si la empresa no reinvierte en mantenimiento.

# <font color="brown">**1. Tu encargo**</font>

No se te pide solo código: se te pide una **recomendación fundamentada**. Al terminar cada bloque deberás entregar dos productos, como haría cualquier asesor de ciencia de datos en un proyecto real de consultoría:

* Un **informe técnico** que recoja el planteamiento del problema, el modelo utilizado, los resultados de la simulación (con sus intervalos de confianza — nunca un único número suelto) y tu interpretación de negocio de cada resultado.
* Una **presentación ejecutiva** pensada para el comité de dirección de "Bio-Distri S.A.": personas que no van a leer tu código ni tus fórmulas, y que necesitan entender, en el menor tiempo posible, cuál es la rentabilidad y el riesgo de la estrategia analizada, y qué recomiendas hacer al respecto.

Al final de cada bloque encontrarás el **encargo concreto de la Dirección**, redactado como una petición formal y organizado por objetivos — esa misma organización por objetivos es la que te conviene usar como esqueleto de tu informe técnico.

# 🟢 <font color="brown">**2. Rentabilidad de un proyecto de mejora**</font>

A continuación se describe la situación actual



## **2.1. El modelo de un proyecto de mejora**

Para cada centro logístico, la empresa selecciona aleatoriamente un tipo de proyecto de mejora ($T$) según la siguiente distribución:

| Proyecto ($T$) | Descripción | Prob. ($p_j$) |
| :--- | :--- | :--- |
| **$T=1$** | Optimización de Cadena de Frío (IoT) | $0.35$ |
| **$T=2$** | IA para Predicción de Demanda | $0.25$ |
| **$T=3$** | Empaques Activos (Bio-polímeros) | $0.25$ |
| **$T=4$** | Auditoría de Procesos y Proveedores | $0.15$ |

Cada proyecto genera tres resultados aleatorios, independientes entre sí **dado $T$**, pero con parámetros que dependen del tipo de proyecto:

1. **Inversión Inicial ($I$):** capital requerido (en miles de €).
2. **Reducción de Merma ($R$):** proporción de desperdicio evitado ($0 \le R \le 1$).
3. **Ahorro Operativo Extra ($S$):** reducción de costes indirectos (en miles de €).

| Proyecto ($T$) | Inversión $I \mid T$ (Lognormal)__________ | Reducción $R \mid T$ (Beta)__________ | Ahorro $S \mid T$ (Gamma)__________ |
| :--- | :--- | :--- | :--- |
| **1. IoT** | $LN(\mu=4,\ \sigma=0.5)$ | $Be(\alpha=5,\ \beta=2)$ | $Ga(k=10,\ \theta=1)$ |
| **2. IA** | $LN(\mu=3.5,\ \sigma=0.8)$ | $Be(\alpha=4,\ \beta=4)$ | $Ga(k=5,\ \theta=2)$ |
| **3. Empaque** | $LN(\mu=3,\ \sigma=0.3)$ | $Be(\alpha=2,\ \beta=5)$ | $Ga(k=2,\ \theta=1)$ |
| **4. Auditoría** | $LN(\mu=2.5,\ \sigma=0.2)$ | $Be(\alpha=2,\ \beta=2)$ | $Ga(k=3,\ \theta=1.5)$ |

> **Nota:** en la Lognormal, $\mu$ y $\sigma$ son la media y la desviación típica del **logaritmo** de la variable (es decir, si $X\sim LN(\mu,\sigma)$ entonces $\ln(X)\sim N(\mu,\sigma)$).

## **2.2. Función de beneficio neto**

El beneficio neto final de un proyecto ($Y$) se define como:

$$Y = (V_{stock}\cdot R) + S - I - \text{Penalización}(R)$$

**Donde:**

* $V_{stock}$ es el valor del stock en riesgo (el género que se salvaría si la merma se redujera al 100%).
* **Penalización regulatoria:** si la reducción de merma $R$ es inferior al **10%** ($R<0.10$), la empresa incurre en una multa por incumplimiento de la normativa de desperdicio alimentario. Si $R\ge 0.10$, la penalización es $0$.

#### 📣 <font color="red">**Nota de unidades.**</font> Para que todas las magnitudes sean directamente comparables, trabajamos **en miles de €**: así, $V_{stock}=500$ (equivalente a $500.000$€) y la multa regulatoria es $\text{Penalización}=20$ (equivalente a $20.000$€) cuando $R<0.10$. Con esta convención, $I$, $S$ e $Y$ resultantes de tu simulador estarán también en miles de €.

## **2.3. Algoritmo de simulación**

$T$, $I$, $R$ y $S$ se simulan mediante el **método de composición**: primero el tipo de proyecto, después las variables condicionadas a él.

1. Simular $t_i\sim T$ (proyectos 1 a 4, con probabilidades $0.35,\ 0.25,\ 0.25,\ 0.15$).
2. Simular, condicionado a $t_i$:
  * $i_i\sim LN(\mu_{t_i},\sigma_{t_i})$,
  * $r_i\sim Be(\alpha_{t_i},\beta_{t_i})$,
  * $s_i\sim Ga(k_{t_i},\theta_{t_i})$ .
3. Calcular la penalización: $20$ si $r_i<0.10$, si no $0$.
4. Calcular el beneficio neto: $y_i = 500\cdot r_i + s_i - i_i - \text{Penalización}(r_i)$.
5. Repetir los pasos 1-4 $nsim$ veces para obtener la muestra $\{y_i\}$.


In [ ]:
#@title **Parámetros del modelo**

# --- Tipos de proyecto y su distribución ---
PROYECTOS = [1, 2, 3, 4]              # 1=IoT, 2=IA, 3=Empaque, 4=Auditoria
PROBS_PROYECTO = [0.35, 0.25, 0.25, 0.15]

# --- Distribuciones condicionadas I|T, R|T, S|T ---
# I ~ Lognormal(mu, sigma); R ~ Beta(a, b); S ~ Gamma(k, theta)
PARAMS_PROYECTO = {
    1: dict(mu_I=4.0, sigma_I=0.5, a_R=5, b_R=2, k_S=10, theta_S=1.0),   # IoT
    2: dict(mu_I=3.5, sigma_I=0.8, a_R=4, b_R=4, k_S=5,  theta_S=2.0),   # IA
    3: dict(mu_I=3.0, sigma_I=0.3, a_R=2, b_R=5, k_S=2,  theta_S=1.0),   # Empaque
    4: dict(mu_I=2.5, sigma_I=0.2, a_R=2, b_R=2, k_S=3,  theta_S=1.5),   # Auditoria
}

# --- Parámetros económicos (en miles de €) ---
VSTOCK = 500.0        # valor del stock en riesgo
PENALIZACION = 20.0   # multa regulatoria si R < UMBRAL_R
UMBRAL_R = 0.10

In [ ]:
#@title **Generador de proyectos**

def _generar_proyectos(T):
    """
    A partir de un vector de tipos de proyecto T ya simulado, genera la
    Inversión (I), la Reducción de merma (R) y el Ahorro operativo (S),
    cada una con la distribución condicionada a su proyecto.

    Esta función es el "núcleo" reutilizado tanto por el simulador del
    Bloque Básico (simulador) como por el del Bloque Avanzado
    (simulador_programa): solo cambia CUÁNTOS proyectos y en qué orden
    se generan, no la forma en que se simulan condicionados a T.
    """
    n = len(T)
    I = np.zeros(n)
    R = np.zeros(n)
    S = np.zeros(n)
    for pj, cfg in PARAMS_PROYECTO.items():
        mask = (T == pj)
        n_j = int(mask.sum())
        if n_j == 0:
            continue
        I[mask] = stats.lognorm.rvs(s=cfg["sigma_I"], scale=math.exp(cfg["mu_I"]), size=n_j)
        R[mask] = stats.beta.rvs(a=cfg["a_R"], b=cfg["b_R"], size=n_j)
        S[mask] = stats.gamma.rvs(a=cfg["k_S"], scale=cfg["theta_S"], size=n_j)
    return I, R, S

In [ ]:
#@title **Simulador**
def simulador(nsim):
    """Simula nsim proyectos de mejora independientes (Bloque Básico)."""
    T = np.random.choice(PROYECTOS, size=nsim, p=PROBS_PROYECTO)
    I, R, S = _generar_proyectos(T)
    penal = np.where(R < UMBRAL_R, PENALIZACION, 0.0)
    Y = VSTOCK * R + S - I - penal
    return pd.DataFrame({"T": T, "I": I, "R": R, "S": S,
                          "Penalizacion": penal, "Y": Y})

Comprobmaos el simulador para conseguir $nsim=10.000$ simulaciones:

In [ ]:
NSIM = 10_000
datos_basico = simulador(NSIM)
datos_basico.head(10)

## **2.4. Verificar funcionamiento del algoritmo**

Antes de empezar a completar las tareas establecidas para analizar el comportamiento del sistema es necesario que verifiques las distribuciones asignadas en la descripción del proceso a partir de las 10000 simulaciones obtenidas.

Para dicha verificación puedes usar la función `gof_continuous` que te premite ajustar y estimar una distribución de tipo continuo  a un conjunto de datos. Para el análsiis de escenario climáticos basta con describir los resulttdos de dicha variable.

Te puedes ayudar tanto de resultados numéricos como de gráficos para verificar las distribuciones asumidas por la empresa y consideradas en el simulador. Presta especial atención a si la media y la varianza muestral de cada variable, calculadas por escenario, son coherentes con los parámetros teóricos.

## **2.5 El encargo de la Dirección**

> **MEMORÁNDUM INTERNO**
>
> **De:** Dirección General, "Bio-Distri S.A."
> **Para:** Equipo de Ciencia de Datos
> **Asunto:** Diagnóstico del riesgo financiero de los proyectos de mejora de calidad
>
> Antes de decidir en qué proyectos seguir invirtiendo, necesitamos entender con números —no con impresiones— la rentabilidad real y el riesgo de cada tipo de proyecto. Trabajad con la muestra de 10.000 proyectos que habéis generado (`datos_basico`) y acompañad **cada estimación de su intervalo de confianza al 95%**: una cifra sin margen de error no nos sirve para tomar una decisión de este calado. Os hemos organizado la petición en tres objetivos.

### Objetivo 1. Cuantificar el riesgo económico de un proyecto de mejora

Necesitamos saber, en cifras concretas, qué podemos esperar de un proyecto cualquiera y hasta qué punto se pueden torcer las cosas en el peor de los casos.

* **O1.1.** Estimad el **beneficio esperado** $E(Y)$, la **probabilidad de pérdida** $Pr(Y<0)$ y el **Análisis de Riesgo Extremo (CVaR)** al 5% (el beneficio promedio en el 5% de los peores proyectos).
* **O1.2.** Calculad el **VaR al 95%** (percentil 5 de $Y$, por macro-réplicas) y comparadlo con el CVaR al 5% del punto anterior. Explicadnos qué información adicional nos aporta el CVaR frente al VaR.
* **O1.3.** Estimad el coeficiente de variación de $Y$ para cada tipo de proyecto por separado. ¿Qué proyecto presenta mayor incertidumbre relativa, y qué implicación tiene eso para nosotros como gestores adversos al riesgo?

### Objetivo 2. Entender qué hay detrás de esa variabilidad

Antes de proponer ninguna solución, queremos saber qué parte de nuestro negocio explica realmente el riesgo.

* **O2.1.** Descomponed, por simulación, la varianza total de $Y$ en la parte atribuible a cada uno de los cuatro términos de $Y=500R+S-I-\text{Penalización}(R)$. ¿Qué término domina la variabilidad de nuestro beneficio?
* **O2.2.** Diseñad un análisis de sensibilidad global que determine cuál de los parámetros del modelo (probabilidades de proyecto, o los parámetros de las distribuciones de $I$, $R$ o $S$) explica mayor proporción de la varianza de $Y$: necesitamos saber dónde concentrar nuestros esfuerzos.

### Objetivo 3. Explorar palancas de mejora del programa de calidad actual

Antes de cambiar de estrategia de raíz, queremos agotar los ajustes más baratos sobre el programa que ya tenemos.

* **O3.1.** Nuestra penalización regulatoria actual es una función escalón (un salto discreto en $R=0.10$). Sustituidla por una función continua y creciente conforme $R$ se aleja de $0.10$ hacia abajo (por ejemplo, proporcional a $(0.10-R)$ cuando $R<0.10$), y contadnos, por simulación, cómo cambiaría la distribución de $Y$ respecto al modelo actual.
* **O3.2.** Planteadnos cómo evaluaríais, por simulación, si nos compensa más mantener la cartera actual de proyectos (con la mezcla de probabilidades dada) o concentrar toda la inversión en los dos proyectos de mayor beneficio esperado (IoT e IA), en términos de beneficio esperado y de riesgo (VaR 95%).
* **O3.3.** El proyecto de Empaques Activos es, de los cuatro, el de peor beneficio esperado. Necesitamos una cifra concreta: ¿cuánto tendría que aumentar el valor del stock en riesgo $V_{stock}$ (por ejemplo, priorizándolo en líneas de producto de mayor valor añadido) para que el proyecto de Empaque alcance por sí solo $E(Y)\ge 0$, manteniendo el resto de condiciones sin cambios?
* **O3.4.** Estamos negociando con el proveedor tecnológico una cláusula de "garantía de resultados": si un proyecto no alcanza el umbral $R\ge 0.10$, el proveedor nos devuelve un 20% de la inversión $I$. Modelad esta cláusula, condicionada a $R<0.10$, y estimad su efecto sobre $E(Y)$ y sobre $Pr(Y<0)$.

# 🔵 3. Un programa de mejora continua a varios años

En la práctica, "Bio-Distri" no financia un proyecto y lo abandona: cada centro logístico se especializa en **un tipo de proyecto** y lo mantiene activo durante varios años, dentro de un programa de mejora continua. Esto introduce dos fenómenos que el Bloque Básico no captura:

* **Curva de aprendizaje:** con la experiencia acumulada, el personal del centro ejecuta cada vez mejor el mismo tipo de proyecto, de modo que la reducción de merma conseguida ($R$) **mejora año a año**, acercándose asintóticamente al 100%.
* **Degradación por falta de mantenimiento:** el ahorro operativo extra ($S$) que aporta un proyecto (por ejemplo, el generado por sensores IoT o por un modelo de IA) **se deteriora con el tiempo** si la empresa no reinvierte en su mantenimiento (calibración de sensores, actualización de modelos, formación del personal). Si la empresa sí reinvierte cada año, el ahorro se mantiene en su nivel de base, pero a cambio de un coste de mantenimiento anual.

La pregunta de negocio central de este bloque es: **¿compensa económicamente invertir cada año en mantenimiento, o es preferible dejar que el ahorro se degrade?**

## 3.1. El modelo del programa plurianual

Cada centro logístico mantiene el **mismo** tipo de proyecto $T$ (simulado igual que en el Bloque Básico) durante un horizonte de $H$ años. Cada año $t=1,\dots,H$ se generan de nuevo, de forma independiente, la inversión $I_t\mid T$, la reducción de merma "cruda" $R^0_t\mid T$ y el ahorro "crudo" $S^0_t\mid T$, **con las mismas distribuciones del Bloque Básico** (reflejando que cada año hay variabilidad de ejecución). Sobre esos valores crudos actúan los dos efectos plurianuales:

**Curva de aprendizaje sobre $R$:**

$$R_t = 1-(1-R^0_t)\cdot\lambda^{\,t-1}$$

donde $0<\lambda<1$ es la **tasa de aprendizaje**. Nótese que en el primer año ($t=1$) se recupera exactamente el modelo del Bloque Básico ($R_1=R^0_1$), y que $R_t\to 1$ a medida que $t$ crece: con suficiente experiencia, el centro logístico llega a evitar casi toda la merma.

**Degradación (o mantenimiento) sobre $S$:**

$$S_t = \begin{cases} S^0_t & \text{si la empresa reinvierte en mantenimiento ese año (coste } C_{mant}\text{)} \\ S^0_t\cdot\delta^{\,t-1} & \text{si la empresa NO reinvierte (sin coste adicional)} \end{cases}$$

con $0<\delta<1$ la **tasa de degradación** anual.

El beneficio neto del año $t$ es análogo al del Bloque Básico, con el coste de mantenimiento restado cuando aplica:

$$Y_t = 500\cdot R_t + S_t - I_t - \text{Penalización}(R_t) - C_{mant}\cdot\mathbb{1}_{\{\text{mantenimiento}\}}$$

Para valorar el programa completo de un centro logístico, se calcula su **Valor Actual Neto (VAN)**, descontando los beneficios futuros a una tasa de descuento anual $r$:

$$VAN = \sum_{t=1}^{H} \frac{Y_t}{(1+r)^{\,t-1}}$$

| Parámetro | Símbolo | Valor |
| :--- | :--- | :--- |
| Horizonte del programa | $H$ | $6$ años |
| Tasa de aprendizaje | $\lambda$ | $0.80$ |
| Tasa de degradación anual (sin mantenimiento) | $\delta$ | $0.85$ |
| Coste de mantenimiento anual | $C_{mant}$ | $3$ (miles €/año) |
| Tasa de descuento anual | $r$ | $0.05$ |

Para comparar de forma precisa la política **"con mantenimiento"** frente a la política **"sin mantenimiento"**, generamos **una única vez** los valores base de cada centro logístico y cada año ($T$, $I_t$, $R^0_t$, $S^0_t$) y aplicamos **ambas** políticas sobre esa misma base. De este modo, la diferencia de VAN entre políticas se debe exclusivamente al efecto de la política, y no al azar de haber usado muestras distintas.

## 3.2. Algoritmo de simulación extendido

1. Para cada uno de los $n\_centros$ centros logísticos, simular su tipo de proyecto $t\sim T$ (una sola vez, válido para los $H$ años).
2. Para cada centro y cada año $t=1,\dots,H$, simular (condicionado al proyecto del centro) $i_t\sim LN$, $r^0_t\sim Be$, $s^0_t\sim Ga$ — esta es la "base común".
3. **Política "con mantenimiento":** calcular $R_t$ con la fórmula de aprendizaje; fijar $S_t=S^0_t$; restar $C_{mant}$ cada año.
4. **Política "sin mantenimiento":** calcular $R_t$ igual que en el paso 3 (la base es la misma); calcular $S_t=S^0_t\cdot\delta^{t-1}$; no restar coste de mantenimiento.
5. En ambas políticas, calcular la penalización de cada año y el beneficio $Y_t$.
6. Calcular el VAN de cada centro logístico, en cada política, descontando los $Y_t$ a la tasa $r$.
7. Repetir los pasos 1-6 muchas veces (macro-réplicas) para estimar por Monte Carlo.

In [ ]:
# @title **Parámetros del programa plurianual**
H = 6                        # horizonte del programa, en años
LAMBDA_APRENDIZAJE = 0.80    # cuanto se reduce la "ineficacia" (1-R0) cada anyo
DELTA_DEGRADACION = 0.85     # decaimiento anual de S si NO hay mantenimiento
C_MANT = 3.0                 # miles euros/anyo, coste de mantenimiento
TASA_DESCUENTO = 0.05        # tasa de descuento anual para el VAN

In [ ]:
#@title **Generador de la base común**
def _generar_base_programa(n_centros):
    """
    Genera la base COMUN de numeros aleatorios para n_centros centros
    logisticos a lo largo de H anyos: el tipo de proyecto T (fijo para
    todo el programa del centro) y, para cada anyo, la inversion I, la
    reduccion de merma "cruda" R0 y el ahorro "crudo" S0 -- generados
    con la MISMA funcion _generar_proyectos del Bloque Basico.

    Esta base se reutiliza sin cambios en las dos politicas (con y sin
    mantenimiento): es la clave de la simulacion pareada.
    """
    T_centro = np.random.choice(PROYECTOS, size=n_centros, p=PROBS_PROYECTO)
    Centro = np.repeat(np.arange(n_centros), H)
    Anio = np.tile(np.arange(1, H + 1), n_centros)
    T = np.repeat(T_centro, H)
    I, R0, S0 = _generar_proyectos(T)
    return pd.DataFrame({"Centro": Centro, "Anio": Anio, "T": T,
                          "I": I, "R0": R0, "S0": S0})

In [ ]:
#@title **Coste programa**
def coste_programa(base, mantenimiento):
    """
    Aplica una politica (mantenimiento=True/False) sobre una base comun
    ya generada, y devuelve el DataFrame completo con R, S, Penalizacion
    e Y de cada centro y cada anyo.
    """
    datos = base.copy()
    datos["R"] = 1 - (1 - datos["R0"]) * (LAMBDA_APRENDIZAJE ** (datos["Anio"] - 1))
    if mantenimiento:
        datos["S"] = datos["S0"]
        coste_mant = C_MANT
    else:
        datos["S"] = datos["S0"] * (DELTA_DEGRADACION ** (datos["Anio"] - 1))
        coste_mant = 0.0
    datos["Penalizacion"] = np.where(datos["R"] < UMBRAL_R, PENALIZACION, 0.0)
    datos["Y"] = (VSTOCK * datos["R"] + datos["S"] - datos["I"]
                  - datos["Penalizacion"] - coste_mant)
    return datos

In [ ]:
#@title **Van por centro**
def van_por_centro(datos):
    """Valor Actual Neto de cada centro logistico, descontando los Y_t."""
    factor_desc = 1 / (1 + TASA_DESCUENTO) ** (datos["Anio"] - 1)
    return (datos["Y"] * factor_desc).groupby(datos["Centro"]).sum()

In [ ]:
#@title **Simulador del programa**
def simulador_programa(n_centros):
    """
    Simula n_centros programas plurianuales completos y devuelve, con
    NUMEROS ALEATORIOS COMUNES, los dos escenarios de politica:
    (datos_con_mantenimiento, datos_sin_mantenimiento).
    """
    base = _generar_base_programa(n_centros)
    d_con = coste_programa(base, mantenimiento=True)
    d_sin = coste_programa(base, mantenimiento=False)
    return d_con, d_sin

Podemos simular del proceso con:

In [ ]:
N_CENTROS = 300
datos_con, datos_sin = simulador_programa(N_CENTROS)
datos_con.head(8)

## **3.3. Verificar funcionamiento del algoritmo**

Antes de empezar a completar las tareas establecidas para analizar el comportamiento del sistema es necesario que verifiques las distribuciones asignadas en la descripción del proceso a partir de las 10000 simulaciones obtenidas.

Para dicha verificación puedes usar la función `gof_continuous` que te premite ajustar y estimar una distribución de tipo continuo  a un conjunto de datos. Para el análsiis de escenario climáticos basta con describir los resulttdos de dicha variable.

Te puedes ayudar tanto de resultados numéricos como de gráficos para verificar las distribuciones asumidas por la empresa y consideradas en el simulador. Presta especial atención a si la media y la varianza muestral de cada variable, calculadas por escenario, son coherentes con los parámetros teóricos.

Obtén la estimación Monte Carlo, con su error, de:

* El **VAN esperado** con y sin mantenimiento: $E(VAN_{con})$, $E(VAN_{sin})$.
* La **ganancia media pareada** por invertir en mantenimiento: $E(VAN_{con}-VAN_{sin})$.
* La **probabilidad de que el mantenimiento compense** en un centro logístico concreto: $Pr(VAN_{con}-VAN_{sin}>0)$.
* El **VaR 95%** (percentil 5) del VAN con mantenimiento, mediante macro-réplicas.

## **3.4 El encargo de la Dirección: ¿compensa mantener los proyectos?**

> **MEMORÁNDUM INTERNO**
>
> **De:** Dirección General, "Bio-Distri S.A."
> **Para:** Equipo de Ciencia de Datos
> **Asunto:** Evaluación del programa de mantenimiento en los centros logísticos
>
> Gracias por el diagnóstico de la fase anterior. Ahora necesitamos que evaluéis si el mantenimiento anual (coste $C_{mant}$) de los proyectos de mejora es una inversión que compensa a lo largo del programa plurianual. Trabajad siempre sobre los registros pareados `datos_con` / `datos_sin` (o el VAN por centro que se deriva de ellos), generados con números aleatorios comunes, y acompañad cada estimación de su intervalo de confianza al 95%. Os hemos organizado la petición en cuatro objetivos, más una ampliación optativa.

### Objetivo 4. Comparar, centro a centro, el programa con y sin mantenimiento

* **O4.1.** Estimad el **VAN esperado** con y sin mantenimiento, $E(VAN_{con})$ y $E(VAN_{sin})$, y la **ganancia media pareada** por mantener, $E(VAN_{con}-VAN_{sin})$. Aprovechad que el diseño es pareado (números aleatorios comunes) para explicarnos por qué el intervalo de confianza de la ganancia sale mucho más estrecho que si hubierais simulado ambas políticas de forma independiente.
* **O4.2.** Calculad la **probabilidad de que el mantenimiento compense** en un centro logístico concreto, $Pr(VAN_{con}-VAN_{sin}>0)$.
* **O4.3.** Complementad el VaR 95% (percentil 5) del VAN con mantenimiento con su **CVaR al 5%**, calculado también para la política sin mantenimiento. ¿Nos protege más el mantenimiento en el centro medio, o en el centro con peor resultado?

### Objetivo 5. Saber de qué depende el valor del mantenimiento

* **O5.1.** Estudiad cómo cambiaría la ganancia media por mantenimiento (O4.1) si la tasa de degradación $\delta$ fuera más leve ($\delta=0.95$) o más severa ($\delta=0.70$), comparando ambos casos con nuestro valor de referencia $\delta=0.85$.
* **O5.2.** Cuantificad el efecto del coste de mantenimiento $C_{mant}$ sobre esa ganancia: repetid el análisis con $C_{mant}$ a la mitad ($1.5$) y al doble ($6$) de nuestro valor de referencia ($3$).
* **O5.3.** Descomponed la ganancia media por mantenimiento según el tipo de proyecto $T$ de cada centro. ¿En qué tipo de proyecto nos resulta más valioso el mantenimiento? Relacionadlo con la magnitud del ahorro $S$ de base en cada tipo de proyecto.

### Objetivo 6. Dimensionar el programa y comprobar su robustez

* **O6.1.** Planteadnos el problema de dimensionamiento: determinad, por simulación, el **coste de mantenimiento máximo** $C_{mant}$ que haría indiferentes ambas políticas ($E(VAN_{con})=E(VAN_{sin})$). Por encima de ese coste, ¿nos seguiría compensando mantener?
* **O6.2.** Nos preocupan los centros con mala suerte prolongada. Diseñad un experimento que fuerce, para un tipo de proyecto concreto, una **racha de 3 años consecutivos** con reducción de merma "cruda" $R^0$ en el percentil 10 inferior de su distribución, y seguid año a año cómo evoluciona el ahorro $S$ y el VAN acumulado bajo cada política, durante la racha y después de ella.
* **O6.3.** Estudiad cómo cambiaría nuestra conclusión sobre el mantenimiento si alargásemos el horizonte del programa $H$ a 10 o 15 años en lugar de los 6 actuales. ¿Por qué cabría esperar que el resultado dependa de $H$?

### Objetivo 7. Vuestra recomendación

* **O7.1.** A partir de los resultados de los Objetivos 4 a 6, redactadnos la recomendación sobre nuestra política de mantenimiento: ¿la recomendaríais con carácter general para toda la red de centros? ¿bajo qué condiciones (de tipo de proyecto, de coste $C_{mant}$, de horizonte $H$) cambiaría vuestra recomendación? Justificadla con números concretos y sus intervalos de confianza, no con impresiones cualitativas.

## Objetivo 8. Ampliación optativa

Lo que sigue **no forma parte del encargo formal** de la Dirección: es un banco de cuestiones adicionales, de mayor dificultad, para quien quiera explorar el modelo del programa plurianual con más profundidad.

* **O8.1.** Diseñad una prueba de hipótesis, o un procedimiento por simulación pareada, para contrastar si la ganancia media por mantenimiento es significativamente distinta de cero al nivel de confianza del 95%.
* **O8.2.** Hemos modelado el programa suponiendo que todos los centros empiezan su curva de aprendizaje desde cero ($t=1$) el primer año. Evaluad, por simulación, si el VAN esperado del programa depende de forma apreciable de si un centro ya tiene experiencia acumulada previa (por ejemplo, empieza el programa en un punto equivalente a $t=3$ de la curva de aprendizaje) frente a empezar completamente desde cero.
* **O8.3.** Diseñad una **política adaptativa** de mantenimiento: en lugar de decidir "mantener siempre" o "no mantener nunca" para todo el horizonte, el centro decide año a año si reinvertir, basándose en si el ahorro $S$ observado ese año cae por debajo de un umbral. Implementad esta política y comparadla, por simulación, con las dos políticas fijas ya estudiadas.
* **O8.4.** Diseñad un esquema de **actualización bayesiana** en el que, tras observar los resultados de los dos primeros años de un centro, actualicéis vuestra creencia sobre la tasa de aprendizaje $\lambda$ de ese centro concreto, y usad esa actualización para decidir si continuar el programa. Describid, a alto nivel, cómo lo implementaríais con simulación.
* **O8.5.** Un fondo público de sostenibilidad nos ofrece cofinanciar el coste de mantenimiento con una subvención de un $X\%$ cada año. Incorporad esta subvención al modelo y estimad, por simulación, a partir de qué porcentaje de cofinanciación el mantenimiento pasaría a compensarnos en promedio.
* **O8.6.** Diseñad una comparación honesta y completa (coste de capital, ganancia operativa, riesgo residual) entre mantener el proyecto original con mantenimiento, frente a sustituirlo, a mitad del horizonte, por un proyecto nuevo del mismo tipo (pagando de nuevo la inversión inicial $I$, pero reiniciando la curva de aprendizaje desde $t=1$). ¿Qué información adicional, no disponible en este simulador, necesitaríais para completar esa comparación?